# 🚀 Training với Mixed Precision (AMP)

Notebook này dùng **`train_amp.py`** thay vì `train.py` để so sánh:

| | `train.py` (baseline) | `train_amp.py` (AMP) |
|--|--|--|
| Precision | float32 | float16 (forward) + float32 (backward) |
| Tốc độ | baseline | ~1.5–2x nhanh hơn |
| VRAM | baseline | ~40% ít hơn |
| Accuracy | baseline | tương đương |
| Output tag | `run_YYYYMMDD_...` | `run_amp_YYYYMMDD_...` |

---
**Cách so sánh:** Chạy cùng số epoch, cùng config → so sánh `timing.json` và V-measure.

In [ ]:
# ==================== CẤU HÌNH ====================
GITHUB_USERNAME = "MoNguyen127"
REPO_NAME       = "radar"
NUM_EPOCHS      = 3
USE_DRIVE       = True
BATCH_SIZE      = 8
WINDOW_LENGTH   = 512   # Giảm từ 1000 xuống 512: tensor N³ từ 2GB → 268MB
                        # (1000→OOM trên T4, 512→OK, 256→thoải mái nhất)
SUBSET_SIZE     = 100000  # None = tất cả windows, int = giới hạn (paper 1 dùng 100000)

## BƯỚC 1: Kiểm tra GPU

In [ ]:
import torch
!nvidia-smi
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    # Kiểm tra AMP support (Tensor Cores)
    cap = torch.cuda.get_device_capability(0)
    amp_supported = cap[0] >= 7  # Volta (V100) hoặc mới hơn
    print(f"Compute capability: {cap[0]}.{cap[1]}")
    print(f"AMP Tensor Core support: {'✓ YES — tốt nhất!' if amp_supported else '⚠ Partial (Pascal, speedup nhỏ hơn)'}")
else:
    print("⚠️ Không có GPU! Enable: Runtime → Change runtime type → GPU")

## BƯỚC 2: Mount Google Drive

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs('/content/drive/MyDrive/radar_deinterleaving', exist_ok=True)
    print('✓ Drive mounted!')

## BƯỚC 3: Clone repo

In [ ]:
import os, shutil, subprocess

os.chdir('/content')

if os.path.exists(REPO_NAME):
    print("Repo exists, pulling latest...")
    os.chdir(REPO_NAME)
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    os.chdir('/content')
else:
    subprocess.run(['git', 'clone', f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'], check=True)

print(f'✓ Repo ready at /content/{REPO_NAME}')

## BƯỚC 4: Cài đặt dependencies

In [ ]:
os.chdir(f'/content/{REPO_NAME}')
subprocess.run(['pip', 'install', '-e', '.', '-q'], check=True)
os.chdir('models_implementation')
subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('✓ Dependencies installed!')
print(f'Working dir: {os.getcwd()}')

## BƯỚC 5: Xác nhận data

In [ ]:
from pathlib import Path

if USE_DRIVE:
    data_dir = Path('/content/drive/MyDrive/radar_deinterleaving/data')
else:
    data_dir = Path('/content/data')

def count_files(path):
    return len(list(path.glob('*.h5'))) if path.exists() else 0

train_count = count_files(data_dir / 'train')
val_count   = count_files(data_dir / 'validation')

print(f'Data dir: {data_dir}')
print(f'Train files:      {train_count}')
print(f'Validation files: {val_count}')

if train_count == 0:
    print('\n⚠️ Không có data! Download trước:')
    print('  import sys; sys.path.insert(0, f"/content/{REPO_NAME}/src")')
    print('  from turing_deinterleaving_challenge import download_dataset')
    print('  download_dataset(save_dir=data_dir, subsets=["train", "validation"])')
else:
    print('✓ Data sẵn sàng!')

## BƯỚC 6: Kiểm tra VRAM trước khi train

Dùng để so sánh với `train.py` baseline.

In [ ]:
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free_vram  = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3
    print(f'Total VRAM: {total_vram:.1f} GB')
    print(f'Free VRAM:  {free_vram:.1f} GB')
    print()
    print('Ước tính VRAM cần:')
    print(f'  train.py  (float32): ~12 GB với batch_size={BATCH_SIZE}, window={WINDOW_LENGTH}')
    print(f'  train_amp (float16): ~7  GB với batch_size={BATCH_SIZE}, window={WINDOW_LENGTH}')
    print(f'  train_amp có thể tăng batch_size lên {int(BATCH_SIZE * 12/7)} mà vẫn an toàn')

## BƯỚC 7: Train với AMP

Dùng `train_amp.py` — output sẽ lưu vào `run_amp_YYYYMMDD_HHMMSS/`

So sánh với baseline `train.py` qua file `timing.json` sau khi xong.

In [ ]:
import time, os, sys

if USE_DRIVE:
    output_dir = '/content/drive/MyDrive/radar_deinterleaving/outputs'
else:
    output_dir = '/content/outputs'

training_cmd = [
    'python', '-u', 'train_amp.py',
    '--data_dir',       str(data_dir),
    '--output_dir',     output_dir,
    '--batch_size',     str(BATCH_SIZE),
    '--num_epochs',     str(NUM_EPOCHS),
    '--learning_rate',  '0.0001',
    '--window_length',  str(WINDOW_LENGTH),
    '--min_emitters',   '2',
    '--validate_every', '1',
    '--save_every',     '1',
    '--num_workers',    '2',
]

if SUBSET_SIZE is not None:
    training_cmd += ['--subset_size', str(SUBSET_SIZE)]

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('Command:', ' '.join(training_cmd))
print('\n🚀 AMP Training bắt đầu...')
print('=' * 60)
sys.stdout.flush()

start_time = time.time()
proc = None
try:
    # Dùng Popen + đọc từng dòng → hiện log real-time như chạy trực tiếp
    proc = subprocess.Popen(
        training_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        universal_newlines=True,
        env=env,
    )
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, training_cmd)

    elapsed = (time.time() - start_time) / 3600
    print(f'\n🎉 TRAINING COMPLETE! Total time: {elapsed:.2f} hours')
except KeyboardInterrupt:
    if proc:
        proc.terminate()
    print('\n⚠️ Interrupted by user')
except Exception as e:
    print(f'\n❌ Training failed: {e}')
    raise

## BƯỚC 8: So sánh kết quả AMP vs Baseline

In [ ]:
import glob, json

# Tìm tất cả runs
all_runs = sorted(glob.glob(f'{output_dir}/run_*'))
amp_runs      = [r for r in all_runs if '/run_amp_' in r]
baseline_runs = [r for r in all_runs if '/run_amp_' not in r and '/run_' in r]

print('=== SO SÁNH AMP vs BASELINE ===')
print()

def load_timing(run_dir):
    timing_file = f'{run_dir}/timing.json'
    if os.path.exists(timing_file):
        with open(timing_file) as f:
            return json.load(f)
    return None

def load_val_result(run_dir):
    val_file = f'{run_dir}/validation_results.json'
    if os.path.exists(val_file):
        with open(val_file) as f:
            return json.load(f)
    return None

for label, runs in [('BASELINE (train.py)', baseline_runs), ('AMP (train_amp.py)', amp_runs)]:
    if not runs:
        print(f'{label}: không có run nào\n')
        continue
    latest = runs[-1]
    print(f'{label}')
    print(f'  Run: {os.path.basename(latest)}')
    timing = load_timing(latest)
    if timing:
        print(f'  Avg epoch time: {timing["avg_epoch_minutes"]:.1f} min')
        print(f'  Total time:     {timing["total_minutes"]:.1f} min')
    val = load_val_result(latest)
    if val:
        print(f'  V-measure:      {val.get("V-measure", "N/A")}')
    print()

## BƯỚC 9: Evaluate model AMP tốt nhất

In [ ]:
amp_runs = sorted(glob.glob(f'{output_dir}/run_amp_*'))
if not amp_runs:
    print('⚠️ Chưa có AMP run nào!')
else:
    latest_amp_run = amp_runs[-1]
    best_model     = f'{latest_amp_run}/best_model.pt'

    if not os.path.exists(best_model):
        print(f'⚠️ best_model.pt không tồn tại tại {latest_amp_run}')
    else:
        print(f'Best model: {best_model}')

        eval_cmd = [
            'python', 'inference.py',
            '--checkpoint', best_model,
            '--data_dir',   str(data_dir),
            '--subset',     'validation',
            '--output_dir', latest_amp_run,
        ]

        print('\n📊 Đánh giá model AMP...')
        subprocess.run(eval_cmd, check=True)

        val_result_file = f'{latest_amp_run}/validation_results.json'
        if os.path.exists(val_result_file):
            with open(val_result_file) as f:
                results = json.load(f)
            print('\n=== KẾT QUẢ AMP MODEL ===')
            for k, v in results.items():
                if isinstance(v, float):
                    print(f'  {k}: {v:.4f}')

## Ghi chú kỹ thuật AMP

### GradScaler là gì?
float16 có range nhỏ hơn float32 → gradient nhỏ có thể bị **underflow về 0**.  
GradScaler nhân loss với scale factor lớn trước backward → gradient đủ lớn để không bị 0.  
Sau đó chia lại trước optimizer.step().

```
loss × scale → backward → gradients × scale → unscale → clip → step
```

### Tại sao forward float16 nhưng backward float32?
- **Forward** (attention, FFN): float16 đủ precision, Tensor Cores nhanh hơn 2-8x
- **Backward** (gradient accumulation): cần float32 để không mất precision qua nhiều layer

### Khi nào AMP KHÔNG hiệu quả?
- GPU cũ (Pascal P100, compute < 7.0): không có Tensor Cores → speedup rất ít
- CPU training: autocast tắt tự động
- Model quá nhỏ: overhead của GradScaler > benefit

### Output tag khác nhau để so sánh:
```
outputs/run_20260320_022119/        ← baseline train.py
outputs/run_amp_20260401_093000/    ← train_amp.py
```